# document-pii-redactor — quickstart

Detect PII in **document images** and **plain text**, then **redact**, **anonymize**, or **de-identify** it.

This notebook walks through every transform on both modalities. Point `IMAGE` at any document image (a lab report, prescription, ID card, …) and run top to bottom.


In [ ]:
%pip install -q "document-pii-redactor[visual]"   # [visual] adds signature/QR/face detection (AGPL-3.0)


Tesseract is only needed for the built-in OCR (skip if you bring your own OCR):
`apt-get install tesseract-ocr` &nbsp;/&nbsp; `brew install tesseract`


## Setup — load once, detect once

`detect()` is the core primitive: it runs OCR + the models a single time and returns structured entities (category, location, text, confidence). Every transform consumes its output.


In [ ]:
from document_pii_redactor import ImagePIIRedactor, TextPIIRedactor

IMAGE = "report.png"          # <- your document image

image_redactor = ImagePIIRedactor("ekacare/document-pii-redactor")
entities = image_redactor.detect(IMAGE)

for e in entities[:10]:
    print(f"{e.kind:6} {e.category:25} {str(e.text)[:40]!r}")
print(f"… {len(entities)} entities total")


In [ ]:
text_redactor = TextPIIRedactor("ekacare/document-pii-redactor")

text = "Mr. John Doe, 45 yrs, DOB 12-03-1979, Indiranagar, Bangalore. Contact: +91 98765 43210."
spans = text_redactor.detect(text)

for sp in spans:
    print(f"{sp.category:22} {sp.text!r}")


## Redact — destroy

Black-out / blur / pixelate image regions; mask text spans. One-way, nothing kept.


In [ ]:
redacted = image_redactor.redact(IMAGE, entities, mode="blur")   # or "solid" / "pixelate"
redacted


In [ ]:
text_redactor.redact(text, spans)


In [ ]:
text_redactor.redact(text, spans, mask="[{category}]")   # mask can name the category


## Anonymize — generalize

One-way, no mapping kept. Ages become 10-year buckets, dates keep only the year, fine geography collapses to `[LOCATION]` (state and country survive), everything else becomes an unnumbered token.


In [ ]:
image_redactor.anonymize(IMAGE, entities)


In [ ]:
text_redactor.anonymize(text, spans)


## De-identify — pseudonymize

Every entity becomes a consistent pseudonym (`Person_1`) — same value, same pseudonym throughout the document — and the entity→pseudonym mapping comes back so an authorized caller can re-link later.


In [ ]:
deid = image_redactor.deidentify(IMAGE, entities)
deid.image


In [ ]:
deid.mapping.entries


In [ ]:
result = text_redactor.deidentify(text, spans)
print(result.text)
result.mapping.entries


Processing multiple pages of the **same record**? Thread the mapping so numbering stays consistent:


In [ ]:
page2 = "Follow-up for Mr. John Doe. Contact +91 98765 43210."
spans2 = text_redactor.detect(page2)
text_redactor.deidentify(page2, spans2, mapping=result.mapping).text   # John Doe is still Person_1


### Hash tokens — stable across documents

`strategy="hash"` derives the pseudonym from the value itself, so the same value gets the same token in **every** document with no mapping to thread. `secret=` salts the hash so guessable values (names, phone numbers) can't be dictionary-reversed.


In [ ]:
text_redactor.deidentify(text, spans, strategy="hash", secret="my-org-salt").text


## Bring your own OCR

Prefer Textract / Google Vision / your own OCR over the built-in Tesseract? Pass the words with their **pixel-coordinate** boxes and OCR is skipped entirely — your exact boxes come back on the detected entities.


In [ ]:
entities_byo = image_redactor.detect(IMAGE,
                                     words=["John", "Doe"],
                                     boxes=[[100, 20, 140, 40], [145, 20, 180, 40]])
[(e.category, e.text, e.bbox) for e in entities_byo]


## Limiting categories

`categories=[...]` on `detect()` limits which of the 53 PII categories are found (default: all).


In [ ]:
only_names = text_redactor.detect(text, categories=["primary_subject_name", "phone_mobile"])
text_redactor.redact(text, only_names)


---

**More**: [GitHub](https://github.com/eka-care/document-pii-redactor) · [live demo](https://huggingface.co/spaces/ekacare/document-pii-redactor) · [model weights](https://huggingface.co/ekacare/document-pii-redactor) · [PyPI](https://pypi.org/project/document-pii-redactor/)
